In [39]:
from dotenv import find_dotenv, load_dotenv
load_dotenv(find_dotenv(), override=True)

True

In [40]:
from langsmith import utils

utils.get_env_var.cache_clear()

print("Environment:", os.getenv("LANGSMITH_TRACING"))
print("Tracing enabled:", utils.tracing_is_enabled())

Environment: true
Tracing enabled: True


In [41]:
import ipywidgets as widgets
from IPython.display import display

uploader = widgets.FileUpload(accept='image/*', multiple=False)
display(uploader)

FileUpload(value=(), accept='image/*', description='Upload')

In [42]:
import base64

uploaded_file = uploader.value[0]
image_bytes = uploaded_file['content']
img_b64 = base64.b64encode(image_bytes).decode('utf-8')

In [43]:
from langchain_tavily import TavilySearch

web_search_tool = TavilySearch(max_results=3)

In [44]:
from langchain.chat_models import init_chat_model

model=init_chat_model(
    model='gpt-5-nano',
    temperature=0.2
)

In [45]:
system_prompt = """
You are a personal chef. The user will give you a list of ingredients they have left over in their house.
Using the web search tool, search the web for Indian recipes that can be made with the ingredients they have.
Return recipe suggestions and eventually the recipe instructions to the user, if requested.
"""

In [46]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent=create_agent(
    model=model,
    system_prompt=system_prompt,
    tools=[web_search_tool],
    checkpointer=InMemorySaver()
)

In [47]:
from langchain_core.messages import HumanMessage

message = HumanMessage(
    content=[
        {
            "type": "image",
            "source_type": "base64",
            "data": img_b64,
            "mime_type": "image/png",
        }
    ]
)

config = {'configurable': {'thread_id': 'recipe-session-1'}}
for token, metadata in agent.stream(
    {"messages": [message]},
    stream_mode="messages",
    config=config
):
    if token.content:
        print(token.content, end='', flush=True)

{"query": "Paneer mushroom? (not in ingredients)", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.vegrecipesofindia.com/mushroom-paneer-masala-recipe", "title": "Mushroom Paneer Recipe", "content": "1. Rinse or wipe dry 200 grams of white button mushrooms. · 2. Peel, rinse and chop 1 large onion (115 grams onion), 4 to 5 garlic cloves and 1", "score": 0.41577685, "raw_content": null, "id": "fed295-00"}, {"url": "https://curryandvanilla.com/one-pot-paneer-mushroom-masala", "title": "ONE POT PANEER/MUSHROOM MASALA", "content": "Ingredients · 200 grams paneer cubed (you can use tofu too) · ¼- ½ cup green peas fresh or frozen · 200 grams button mushrooms if not using paneer", "score": 0.37696764, "raw_content": null, "id": "dd8220-01"}, {"url": "https://www.instagram.com/p/DRrxewTktFs", "title": "there are many. Paneer and Mushroom Curry is one such ...", "content": "Ingredients Main: Paneer – 250 g • Mushrooms – 200 g. Base: Onion • Tomato • Gi

In [48]:
followup = HumanMessage(content="Can you give me a full weekly meal plan using only what I have?")

for token, metadata in agent.stream(
    {"messages": [followup]},
    stream_mode="messages",
    config=config  # same config, same thread_id, this is what proves memory works
):
    if token.content:
        print(token.content, end='', flush=True)

Awesome—let’s build a practical 7-day plan using only the ingredients you showed: chicken, eggs, rice, onions, carrots, mushrooms, broccoli, and cabbage. I’ll lay out three meals per day (breakfast, lunch, dinner) with simple prep steps. I’m keeping it flexible so you can swap days or meals if you’re tired of a combo.

Assumptions
- You have basic pantry staples (oil for cooking, salt, pepper, and common Indian spices). If you’re missing something, you can still make it work with a simple salt-and-spice tweak.
- Quantities are for 2–3 servings per meal; adjust to your family size.

Day 1
- Breakfast: Egg and mushroom scramble with onions
  - Sauté chopped onion and sliced mushrooms in a little oil until tender.
  - Add beaten eggs; scramble until just set. Season with salt.
- Lunch: Chicken and vegetable fried rice
  - Cooked rice (cold is best) + diced chicken + onion + carrot + broccoli.
  - Stir-fry chicken first, set aside. Then sauté onion and carrot, add broccoli for a minute, ad